In [1]:
import torch

In [2]:
import gpuclustersim

In [4]:
device = "gpuclustersim"

In [5]:
x = torch.randn(10, device=device)
y = torch.randn(10, device=device)
z = x + y

[GCS] Fallback: aten::normal_ | in0:[10] | in1:scalar | in2:scalar
[GCS] Fallback: aten::normal_ | in0:[10] | in1:scalar | in2:scalar
[GCS] Fallback: aten::add | in0:[10] | in1:[10] | in2:scalar | in3:[10]


In [7]:
x = torch.tensor([[1,2], [3,4]], device=device)

In [8]:
y = (x*x+x*x)/x**2

[GCS] Fallback: aten::mul | in0:[2, 2] | in1:[2, 2] | in2:[2, 2]
[GCS] Fallback: aten::mul | in0:[2, 2] | in1:[2, 2] | in2:[2, 2]
[GCS] Fallback: aten::add | in0:[2, 2] | in1:[2, 2] | in2:scalar | in3:[2, 2]
[GCS] Fallback: aten::pow | in0:[2, 2] | in1:scalar | in2:[2, 2]
[GCS] Fallback: aten::div | in0:[2, 2] | in1:[2, 2] | in2:[2, 2]


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("gpuclustersim")

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 20)
        self.fc2 = nn.Linear(20, 20)
        self.fc3 = nn.Linear(20, 1)
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

In [10]:
model = Net().to(device)

In [11]:
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

In [12]:
for step in range(10):
    x = torch.randn(32, 10).to(device)
    y = torch.randn(32, 1).to(device)
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()

[GCS] Fallback: aten::addmm | in0:[20] | in1:[32, 10] | in2:[10, 20] | in3:scalar | in4:scalar | in5:[32, 20]
[GCS] Fallback: aten::relu | in0:[32, 20]
[GCS] Fallback: aten::addmm | in0:[20] | in1:[32, 20] | in2:[20, 20] | in3:scalar | in4:scalar | in5:[32, 20]
[GCS] Fallback: aten::relu | in0:[32, 20]
[GCS] Fallback: aten::addmm | in0:[1] | in1:[32, 20] | in2:[20, 1] | in3:scalar | in4:scalar | in5:[32, 1]
[GCS] Fallback: aten::mse_loss | in0:[32, 1] | in1:[32, 1] | in2:scalar | in3:[]
[GCS] Fallback: aten::fill_ | in0:[] | in1:scalar


/Users/yassawekainolda/projects/ML/gpuclustersim/.venv/lib/python3.10/site-packages/torch/nn/functional.py:4315: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [32, 1]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:37.)
  return torch._C._nn.mse_loss(


RuntimeError: Please register PrivateUse1HooksInterface by `RegisterPrivateUse1HooksInterface` first.